# Notebook 01: Bronze Ingestion
## Propósito
Ingestar los 8 CVS de Olist desde "Files/raw/ del lakehouse Bronze y persistirlos como tablas Delta. Sin transformaciones logicas
## Inputs
- 8 CSV en "{BRONZE_LAKEHOUSE}/Files/{RAW_FILES_PATH}/
## Outputs
- 8 tablas Delta "bronze_*" en {BRONZE_LAKEHOUSE}
## Parametrización
Ver cerda 2. Defaults para distintos ambientes

In [12]:
# =================================================================
# PARÁMETROS DEL NOTEBOOK
# =================================================================
# Estos valores se pueden sobrescribir cuando el notebook es invocado
# desde un Data Pipeline. Si se corre manualmente, usa estos defaults.
# -----------------------------------------------------------------

#Lakehouse donde se escribieron las tablas Bronze
BRONZE_LAKEHOUSE = "lh_olist_bronze"

#Ruta relativa dentro de Files/ 
RAW_FILES_PATH = "raw"

# Modo de escritura: "overwrite"
WRITE_MODE = "overwrite"

StatementMeta(, 27358c5e-7987-4ccf-ade2-31b99341334c, 13, Finished, Available, Finished, False)

In [13]:
## **Celda 3 — Imports:**

from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

print(f"Inicio: {datetime.now()}")
print(f"Bronze lakehouse: {BRONZE_LAKEHOUSE}")
print(f"Raw files path: Files/{RAW_FILES_PATH}")
print(f"Write mode: {WRITE_MODE}")

StatementMeta(, 27358c5e-7987-4ccf-ade2-31b99341334c, 14, Finished, Available, Finished, False)

Inicio: 2026-05-30 17:38:30.443080
Bronze lakehouse: lh_olist_bronze
Raw files path: Files/raw
Write mode: overwrite


In [14]:
## **Celda 4 — Helpers y catalogo de archivos:**

ARCHIVOS_RAW = {
    "olist_customers_dataset.csv":"bronze_customers",
    "olist_orders_dataset.csv":"bronze_orders",
    "olist_order_items_dataset.csv":"bronze_order_items",
    "olist_order_payments_dataset.csv":"bronze_order_payments",
    "olist_order_reviews_dataset.csv":"bronze_order_reviews",
    "olist_products_dataset.csv":"bronze_products",
    "olist_sellers_dataset.csv":"bronze_sellers",
    "product_category_name_translation.csv":"bronze_category_translation",
}

def read_raw_csv(filename: str):
    "lee un CSV de Files/{RAW_FILES_PATH}/ del lakehouse Bronze"
    return(spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("multiline", "true") #r saltos de linea
            .option("escape", '"') #r comas fuera
            .load(f"Files/{RAW_FILES_PATH}/{filename}"))

def write_bronze(df, table_name: str):
    "Escribe un Dataframe como tabla Delta en Bronze"
    (df.write.format("Delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BRONZE_LAKEHOUSE}.{table_name}"))

def read_bronze(table_name: str):
    "Helper para validar la carga"
    return spark.read.table(f"{BRONZE_LAKEHOUSE}.{table_name}")

StatementMeta(, 27358c5e-7987-4ccf-ade2-31b99341334c, 15, Finished, Available, Finished, False)

In [15]:
## **Celda 5 — Ingesta:**

print("=" * 60)
print("INGESTA BRONZE")
print("=" * 60)

resultados = {}

for archivo, tabla in ARCHIVOS_RAW.items():
    print(f"{archivo} - {tabla}")
    df = read_raw_csv(archivo)
    df_con_audit = (df
                    .withColumn("_ingestion_timestamp", current_timestamp())
                    .withColumn("_source_file", lit(archivo)))
    write_bronze(df_con_audit, tabla)
    count = read_bronze(tabla).count()
    resultados[tabla] = count
    print(f"{count:,} filas")
print(f"Total filas ingestadas {sum(resultados.values()):,}")


StatementMeta(, 27358c5e-7987-4ccf-ade2-31b99341334c, 16, Finished, Available, Finished, False)

INGESTA BRONZE
olist_customers_dataset.csv - bronze_customers
99,441 filas
olist_orders_dataset.csv - bronze_orders
99,441 filas
olist_order_items_dataset.csv - bronze_order_items
112,650 filas
olist_order_payments_dataset.csv - bronze_order_payments
103,886 filas
olist_order_reviews_dataset.csv - bronze_order_reviews
99,224 filas
olist_products_dataset.csv - bronze_products
32,951 filas
olist_sellers_dataset.csv - bronze_sellers
3,095 filas
product_category_name_translation.csv - bronze_category_translation
71 filas
Total filas ingestadas 550,759


In [16]:
## **Celda ASSERT 6 — Validacion de conteos vs valores esperados:** 

ESPERADOS = {
    "bronze_customers":(99_000, 100_000),
    "bronze_orders":(99_000, 100_000),
    "bronze_order_items":(112_000, 113_000),
    "bronze_order_payments":(103_000, 105_000),
    "bronze_order_reviews":(98_000, 100_000),
    "bronze_products":(32_000, 34_000),
    "bronze_sellers":(3_000, 3_200),
    "bronze_category_translation":(60, 80),
}

print("=" * 60)
print("VALIDACIÓN DE CONTEOS")
print("=" * 60)

problemas = 0

for tabla, (low, high) in ESPERADOS.items():
    actual = resultados.get(tabla, 0)
    ok = low <= actual <= high
    status = "Correcto" if ok else "Incorrecto"
    print(f"{status} {tabla}: {actual:,} (esperados entre {low:,} y {high:,})")
    if not ok:
        problemas += 1

if problemas == 0:
    print(f"Ingesta de Bronze completa de manera correcta")
else:
    print(f"{problemas} tablas con conteos fuera del rango")
    raise Exception("Validación Bronze fallida")

StatementMeta(, 27358c5e-7987-4ccf-ade2-31b99341334c, 17, Finished, Available, Finished, False)

VALIDACIÓN DE CONTEOS
Correcto bronze_customers: 99,441 (esperados entre 99,000 y 100,000)
Correcto bronze_orders: 99,441 (esperados entre 99,000 y 100,000)
Correcto bronze_order_items: 112,650 (esperados entre 112,000 y 113,000)
Correcto bronze_order_payments: 103,886 (esperados entre 103,000 y 105,000)
Correcto bronze_order_reviews: 99,224 (esperados entre 98,000 y 100,000)
Correcto bronze_products: 32,951 (esperados entre 32,000 y 34,000)
Correcto bronze_sellers: 3,095 (esperados entre 3,000 y 3,200)
Correcto bronze_category_translation: 71 (esperados entre 60 y 80)
Ingesta de Bronze completa de manera correcta
